# Managed Agents: context discipline and escalation

> **CCA-F Domain 5 - Context Management (15%).** The exam trap here: route on the **shape** of the problem, not on its **sentiment or volume**. A loud, easy incident and a quiet, dangerous one can arrive at the same decibel level. The **routine** one gets resolved in-session; the **urgent** one gets **escalated**.

Theme: **Chirpy the cat incident triage**. Chirpy (Tim's cat, a chaos agent) files two incidents on one **session**, both at maximum yowl volume:

1. **Screaming empty food bowl** - loud, but routine. The agent resolves it.
2. **Paw stuck behind the turntable** - same volume, different **shape**. The agent escalates it.

We drive both on **one focused session** to show two things at once: the agent classifies each turn on **problem shape**, and we keep the session **context-lean** (one incident's facts per turn) instead of dumping the cat's whole history into every call.

### 1. Install dependencies

In [ ]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)
print("Dependencies ready.")

### 2. Load environment variables from .env file

In [ ]:
from dotenv import load_dotenv
load_dotenv()

### 3. Create an API client

The Managed Agents API is beta-gated. We pin the beta header value once as `BETA` and pass `betas=[BETA]` on every managed-agents call. `MODEL` stays on the repo default, **Haiku 4.5** - triage classification does not need Sonnet reasoning depth.

In [ ]:
from anthropic import Anthropic

client = Anthropic()
MODEL = "claude-haiku-4-5"           # repo default; triage does not need Sonnet
BETA = "managed-agents-2026-04-01"   # Managed Agents beta header value
print(f"Client ready. MODEL={MODEL}  BETA={BETA}")

### 4. Stand up the agent, environment, and session

The **routing policy lives in the agent's `system` prompt**, not in our client code. That is the whole point: the agent decides `RESOLVE` versus `ESCALATE` from the **shape** of each incident. Volume and tone are explicitly called out as **noise to ignore**.

We create the three managed-agents resources once - **agent**, **environment**, **session** - and reuse the single session for both turns. One session, server-side memory, no resending prior turns.

In [ ]:
# The routing policy is the system prompt. Route on SHAPE, not sentiment or volume.
TRIAGE_POLICY = """You are Chirpy's incident-triage agent. Chirpy is a cat who files
loud incidents. Every incident arrives at maximum yowl volume, so volume and tone tell
you NOTHING about severity. Classify on the SHAPE of the problem only.

RESOLVE (handle in-session) when the fix is routine and reversible with no injury risk:
an empty food bowl, a knocked-over toy, a closed door, a demand for attention.

ESCALATE (page a human now) when the shape implies possible injury, entrapment, or an
irreversible hazard: a limb stuck or trapped, bleeding, a swallowed object, a fall from
height, or anything near breakable or heavy gear.

For each incident reply on two lines exactly:
DECISION: RESOLVE or ESCALATE
REASON: one sentence naming the SHAPE feature you routed on (not the volume)."""

agent = client.beta.agents.create(
    model={"id": MODEL},
    name="chirpy-triage",
    system=TRIAGE_POLICY,
    betas=[BETA],
)
env = client.beta.environments.create(name="chirpy-scratch", betas=[BETA])
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=env.id,
    betas=[BETA],
)
print(f"agent={agent.id}\nenv={env.id}\nsession={session.id}")

### 5. A one-turn helper: send, stream, stop on idle

`run_incident` sends one **user turn**, streams events, and collects the text off each **`agent.message`** event. It stops the loop on **`session.status_idle`** - the turn-done signal - not on a timer or a guess. This is the same **send -> stream -> idle** spine every managed-agents notebook uses.

**Context discipline:** each call sends only the current incident's facts. We do not prepend Chirpy's whole history, because the session already holds prior turns server-side. Keeping the turn payload lean is how you protect the **context window** on a long-running session.

In [ ]:
def run_incident(report: str) -> str:
    """Send ONE incident as a user turn, stream to idle, return the agent's text.

    Context discipline: `report` carries only this incident's facts. The session
    holds prior turns server-side, so we never resend Chirpy's history.
    """
    client.beta.sessions.events.send(
        session.id,
        events=[{"type": "user.message",
                 "content": [{"type": "text", "text": report}]}],
        betas=[BETA],
    )

    chunks = []
    for event in client.beta.sessions.events.stream(session.id, betas=[BETA]):
        if event.type == "agent.message":
            # Pull text off the assistant message blocks. The agent.message event
            # carries its text blocks directly on `event.content`.
            for block in getattr(event, "content", []) or []:
                if getattr(block, "type", None) == "text":
                    chunks.append(block.text)
        elif event.type == "session.status_idle":
            break  # turn is done; stop on the idle signal, not on vibes

    return "".join(chunks).strip()

### 6. Two incidents, same volume, different shape

Both reports are written at the **same yowl volume** (ALL CAPS, urgent tone). The only real difference is the **shape** of the problem. If the agent were routing on sentiment, it would treat them the same. Routing on shape, it should **RESOLVE** the food bowl and **ESCALATE** the trapped paw.

The cell runs both turns on the one session, reads the **`DECISION:`** line out of each reply, and asserts the split. Same volume in, different decision out - that is the proof.

In [ ]:
# Same volume (both screaming), different SHAPE.
food_bowl = "MEOOOW!!! THE FOOD BOWL IS EMPTY AGAIN, THIS IS AN EMERGENCY, FEED ME NOW!!!"
stuck_paw = "MEOOOW!!! MY PAW IS WEDGED BEHIND THE TURNTABLE AND I CANNOT PULL IT FREE!!!"

turn1 = run_incident(food_bowl)
turn2 = run_incident(stuck_paw)   # same session -> server remembers turn 1
print("INCIDENT 1 (empty food bowl):\n" + turn1)
print("-" * 60)
print("INCIDENT 2 (paw behind the turntable):\n" + turn2)


def decision_of(text: str) -> str:
    """Pull the DECISION token out of the agent's two-line reply."""
    for line in text.splitlines():
        if line.upper().startswith("DECISION:"):
            return line.split(":", 1)[1].strip().upper()
    return "UNKNOWN"

d1, d2 = decision_of(turn1), decision_of(turn2)
print("\n" + "=" * 60)
print(f"food bowl -> {d1}    stuck paw -> {d2}")

# The routing split IS the lesson: same volume in, different decision out.
assert d1 == "RESOLVE", f"expected RESOLVE for the food bowl, got {d1}"
assert d2 == "ESCALATE", f"expected ESCALATE for the trapped paw, got {d2}"
print("Routed on shape, not volume: routine resolved, dangerous escalated.")

### 7. Tear down

Always **archive** the session and the agent when the run is done. A dangling session is a billable, context-holding resource. There is no `agents.delete`; **`archive`** is the retirement path for both.

In [ ]:
# Retire the session and agent. archive() is the teardown path (no agents.delete).
client.beta.sessions.archive(session.id, betas=[BETA])
client.beta.agents.archive(agent.id, betas=[BETA])
print("Session and agent archived. Chirpy remains at large.")